In [1]:
import yfinance as yf
import pandas as pd

# Global multi-asset universe containing Crypto, Commodities, Fixed Income, and Equities
tickers = [
    "BTC-USD", "ETH-USD", "SOL-USD", "LINK-USD", "GC=F", "SI=F", "BZ=F", "NG=F", "HG=F", 
    "ZC=F", "KC=F", "PA=F", "TLT", "IEF", "SHY", "TIP", "BNDX", "EMB", "VTC", "JNK", 
    "IBGL.L", "MUB", "AAPL", "MSFT", "AMZN", "JNJ", "JPM", "XOM", "PG", "TSLA", 
    "UNH", "BRK-B", "SAN.MC", "ITX.MC", "IBE.MC", "MC.PA", "SAP.DE", "ASML.AS", 
    "SIE.DE", "NESN.SW", "AZN.L", "HSBA.L", "2330.TW", "7203.T", "BABA", 
    "TCEHY", "RELIANCE.NS", "VALE", "BHP"
]

# Accumulator list for parsed asset data dictionaries before converting to DataFrame
rows = []

# --- Data Extraction Loop ---
for ticker in tickers:
    # Fetch raw metadata dictionary using yfinance API
    info = yf.Ticker(ticker).info

    fcf = info.get("freeCashflow") 
    market_cap = info.get("marketCap")

    # Calculate Free Cash Flow (FCF) Yield to evaluate capital efficiency relative to valuation
    # Safely handles missing/null financial variables from the API payload
    if fcf and market_cap:
        fcf_yield = fcf / market_cap
    else:
        fcf_yield = None

    # Map unstructured API fields to standardized fundamental and metadata schema layout
    rows.append({
        "ticker": ticker, 
        "asset_name": info.get("shortName"), 
        "sector": info.get("sector"), 
        "exchange": info.get("exchange"), 
        "currency": info.get("currency"), 
        "beta": info.get("beta"), 
        "dividend_yield": info.get("dividendYield"), 
        "trailingpe": info.get("trailingPE"), 
        "pricetobook":  info.get("priceToBook"), 
        "fcf_yield": fcf_yield, 
        "revenuegrowth": info.get("revenueGrowth"), 
        "earningsgrowth": info.get("earningsGrowth"), 
        "forwardeps": info.get("forwardEps"), 
        "payoutratio": info.get("payoutRatio") 
    })

# Convert raw dictionary sequences into a structured tabular format
assets_df = pd.DataFrame(rows)


def asset_type(ticker):
    """Determines the broad asset class categorization based on predefined ticker lists.

    Args:
        ticker (str): The unique financial instrument identifier code.

    Returns:
        str: The designated asset type identifier ('Crypto', 'Commodity', 'Bond', or 'Equity').
    """
    crypto = ["BTC-USD", "ETH-USD", "SOL-USD", "LINK-USD"]
    
    commodities = ["GC=F", "SI=F", "BZ=F", "NG=F", 
                   "HG=F", "ZC=F", "KC=F", "PA=F"]
    
    bonds = ["TLT", "IEF", "SHY", "TIP", "BNDX", "EMB", 
             "VTC", "JNK", "IBGL.L", "BTP.MI", "MUB"]
    
    if ticker in crypto:
        return "Crypto"
    
    elif ticker in commodities:
        return "Commodity"
    
    elif ticker in bonds:
        return "Bond"
    
    else:
        return "Equity"


# Apply deterministic classification criteria row-wise across the asset index
assets_df["asset_type"] = assets_df["ticker"].apply(asset_type)

# Export final cleaned metrics layout. Index is omitted to maintain tabular structure continuity
assets_df.to_csv("data/assets_info.csv", index=False)